# Temporal averaging of the coastal boundary

MODFLOW~6 integrates a whole coupling interval in one backward-Euler step, and that
step applies the boundary flux across the entire interval: the volume exchanged is
$\Delta t\,C\,(H-h)$. So the boundary head $H$ has to represent the interval, not the
instant at its end. Those coincide only while the boundary varies slowly, and a tidal
boundary sampled over an appreciable fraction of a tidal cycle does not.

Two reductions of the D-Flow FM stage and depth to that single boundary value are
compared over coupling intervals from 30 minutes to one day, scored against a
15-minute reference. Every interval divides a day, so that coupling steps land on the
daily stress-period boundaries; the coarse grid adds 6 and 12 hours to the six run on
all three. The comparison is run on all three grids, so a conclusion that holds on
one and not on the others is visible as such:

- **instant** -- the value at the end of the interval. Every scenario before August
2026 used this.
- **mean** -- a wetted-fraction weighted time average. For the GHB this is exact
rather than approximate: the conductance is $C_0$ times a binary wet mask, so the
interval-mean flux factors as $C_0\langle w\rangle(\langle w s_1\rangle/\langle
w\rangle - h)$, making the wetted fraction the conductance multiplier and the stage a
wet-weighted mean.

## Result

The two methods fail in opposite directions, and which one wins is decided by the
tide, not by the model:

- **instant** never damps amplitude, but aliases once the interval exceeds the
Nyquist limit for the dominant constituent.
- **mean** never aliases, but damps amplitude by an amount that grows with the
interval.

Both penalties are visible in the aquifer head, and they divide the interval range
between them. Below 8 hours the average is the worse of the two at fourteen of the
fifteen grid-and-interval combinations, by as much as a factor of 2.3, which is the
damping penalty; at daily coupling instantaneous sampling is worse by an order of
magnitude, which is the aliasing penalty.

The sampled error is not monotonic in the coupling interval. On the coarse grid it is
1.3 mm at 8 hours, 36.9 mm at 12 hours, and 32.9 mm at 24 hours, so halving the
interval from a day to twelve hours makes the solution worse rather than better.
Sampling every 12 and every 24 hours folds $M_2$ onto the same 14.8-day beat and the
two carry comparable error, while sampling every 8 hours folds it onto a 22.5-hour
beat that the aquifer damps, and costs almost nothing although 8 hours is already
above the Nyquist limit. The averaged error over the same range grows smoothly from
0.05 to 2.7 mm, as a truncation error does. What sets the sampled error is the period
onto which the constituent aliases, not the length of the interval and not the
crossing of the limit in itself.  The daily separation is large and grid-independent.
Aquifer head RMSE at a one-day interval is 32.9 mm under instantaneous sampling
against 2.7 mm under the average on the coarse grid, 38.8 against 3.4 mm on the
midres grid, and 31.4 against 3.0 mm on the highres grid, factors of 12.0, 11.4 and
10.4. Three independently gridded solutions agree on that factor to within 14
percent. Sewer seepage behaves the same way, favoring the average at daily coupling
by factors of 1.4, 2.9 and 2.7.

At every interval of 8 hours or less both errors stay under 2.2 mm, so the damping
penalty is real but carries no physical consequence at this scale. The choice of
reduction matters at daily coupling and nowhere else.

## What the tracer does not show

The sewer tracer was previously reported through its peak concentration, a maximum
over the whole space-time field, and that statistic does not describe the domain. On
the coarse grid the maximum falls in cell 1669 in all twelve runs, at 7.9 times the
99.99th percentile of the same field; on the highres grid it falls in one of two
cells, at 54 times. The +27 % at 8 hours and +62 % at daily coupling once quoted from
it are that one cell moving.

Neither of the other grids reproduces them. On the midres grid the maximum is twice
the 99.99th percentile, not an outlier at all, and it moves by less than four tenths
of a percent at every interval under both reductions. The coarse spike is one cell,
on one grid.

Measured across the field the tracer is silent. The 99.99th percentile stays within 1
percent of the reference on the coarse grid and within a third of a percent on the
midres grid, and the 99.9th percentile within six tenths of a percent on both. The
highres field sits 3 to 4 percent off the reference at 1, 2, 4 and 8 hours, intervals
at which the head error is below a millimetre and the coupled solution has converged,
so that offset is a noise floor rather than a response to the reduction. Only the
highres daily point, 18 percent against 12 for the average, stands clearly above it,
and it is the one tracer number on any grid that does.

The tracer therefore neither supports nor contradicts the head result: at these
coupling intervals it is too insensitive to distinguish the two reductions at all.
The argument rests on the head and the seepage, which agree across three grids. The
abrupt onset at the Nyquist limit that the peak appeared to show is not a property of
the tracer field.

In [ ]:
%matplotlib inline
import pathlib as pl
import sys

import flopy.plot.styles as styles
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
ROOT = pl.Path.cwd().parent
FIGS = ROOT / "docs" / "GP" / "figures"

# Which sweep to score. Nothing else about the comparison is set here: the run names,
# the reference, and the statistics all live in the module loaded below, because a
# second copy of them here is exactly how this notebook came to disagree with it.
# The interval list is not set here either; it is read off the archive, since the
# coarse grid carries 6 h and 12 h and the other two do not.
GRID = "coarse"          # "coarse", "medium" or "high"


In [ ]:
# The statistics and the figure each have one implementation, in
# docs/GP/scripts/. Duplicating them here would let the notebook and the manuscript
# drift apart, which is how the sequence figure once came to disagree with the text
# describing it.
sys.path.insert(0, str(ROOT / "docs" / "GP" / "scripts"))
import boundary_averaging_data as bad

ds, source = bad.load_or_refresh(grid=GRID)
print(f"{GRID}: " + ("recomputed from results/" if source == "results"
                     else f"read archive; {len(bad.missing(grid=GRID))} runs absent"))
stats = ds.to_dataframe().reset_index()
stats["hours"] = stats["interval"].map(dict(zip(ds.interval.values, ds.hours.values)))

# Taken from the archive rather than spelled out. Hard-coding the labels is what
# silently emptied these tables when the module moved its reference from the
# 30-minute run to the 15-minute one, and a hard-coded interval list would now drop
# the two the coarse grid has and the others do not.
ORDER = list(stats.sort_values("hours")["interval"].unique())
REFS = list(ds.ref.values)
REF0 = REFS[0]
PEAK_REF = float(ds.attrs["peak_reference_concentration"])
P9999_REF = float(ds.attrs["p9999_reference_concentration"])
P999_REF = float(ds.attrs["p999_reference_concentration"])
NYQUIST_H = float(ds.attrs["m2_nyquist_hours"])
print(f"references: {REFS}")
print(f"intervals: {ORDER}")
print(f"Nyquist limit for M2: {NYQUIST_H:.2f} h")
print(f"reference tracer  max {PEAK_REF:.4f}   99.99th {P9999_REF:.5f}   "
      f"99.9th {P999_REF:.5f}")

In [ ]:
for rlabel in REFS:
    s = stats[stats["ref"] == rlabel].set_index("interval").loc[ORDER]
    print(f"\n=== reference: {rlabel} ===")
    print("ratio = instant RMSE / mean RMSE;  > 1 means the mean is closer\n")
    print(s[["hours", "head_inst", "head_mean", "head_ratio",
             "seep_inst", "seep_mean", "seep_ratio",
             "trac_inst", "trac_mean", "trac_ratio"]]
          .rename(columns={"head_inst": "head_i(mm)", "head_mean": "head_m(mm)"})
          .round(4).to_string())

# peak is a max over the whole space-time field, and a max is the least robust
# statistic there is: on coarse it lands in cell 1669 in all twelve runs at 7.9x the
# 99.99th percentile, and on highres in one of two cells at 54x. So it is reported
# beside the percentiles, which no single cell can move. Where the two disagree the
# peak is describing a hot spot, not the domain.
s = stats[stats["ref"] == REF0].set_index("interval").loc[ORDER]
amp = pd.DataFrame({
    "peak i%": 100 * (s.peak_inst - PEAK_REF) / PEAK_REF,
    "peak m%": 100 * (s.peak_mean - PEAK_REF) / PEAK_REF,
    "p99.99 i%": 100 * (s.p9999_inst - P9999_REF) / P9999_REF,
    "p99.99 m%": 100 * (s.p9999_mean - P9999_REF) / P9999_REF,
    "p99.9 i%": 100 * (s.p999_inst - P999_REF) / P999_REF,
    "p99.9 m%": 100 * (s.p999_mean - P999_REF) / P999_REF,
})
print(f"\n\ntracer amplitude, percent departure from the {REF0} reference")
print(amp.round(1).to_string())

In [ ]:
# The figure is drawn by docs/GP/scripts/make_boundary_averaging_figure.py from the
# archive that load_or_refresh wrote above, not from the simulation output: that
# output is tens of gigabytes and is not in version control, so a co-author with only
# the repository could not otherwise rebuild a manuscript figure. This cell used to
# rebuild the summary and write a second archive of its own, to a different path than
# the module's -- which is the drift the single implementation exists to prevent.
import make_boundary_averaging_figure as mbaf

mbaf.make(GRID)
print("figure:", mbaf.output_path(GRID))

In [ ]:
# Kept for interactive use; the manuscript figure is the one written above. Panel D
# carries the 99.99th percentile rather than the max, so this view shows the
# field-wide amplitude where the manuscript figure shows the hot-spot one.
C_I, C_M = "#d62728", "#1f77b4"
s = stats[stats["ref"] == REF0].sort_values("hours")
h = s["hours"].to_numpy()
# Ticks follow the archive, so adding an interval does not silently drop a point.
TICK_LAB = [f"{v * 60:.0f} min" if v < 1 else f"{v:.0f} h" if v < 24
            else f"{v / 24:.0f} d" for v in h]

with styles.USGSPlot():
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(7.5, 5.2), layout="constrained")
    panels = [(axs[0, 0], "head_inst", "head_mean", "Aquifer head RMSE, in millimeters", True),
              (axs[0, 1], "seep_inst", "seep_mean", "Sewer seepage RMSE, in cubic feet per day", True),
              (axs[1, 0], "trac_inst", "trac_mean", "Sewer tracer RMSE, dimensionless", True)]
    for i, (ax, ci, cm, lab, logy) in enumerate(panels):
        ax.plot(h, s[ci], "o-", color=C_I, lw=1.2, ms=4, label="instantaneous")
        ax.plot(h, s[cm], "s-", color=C_M, lw=1.2, ms=4, label="time-averaged")
        ax.set_xscale("log")
        if logy:
            ax.set_yscale("log")
        ax.axvline(NYQUIST_H, color="0.35", lw=0.9, linestyle=(0, (3, 2)), zorder=1)
        ax.set_xticks(h)
        ax.set_xticklabels(TICK_LAB, fontsize=7)
        ax.tick_params(labelsize=7, top=False)
        styles.heading(ax=ax, letter="ABCD"[i], heading=lab, fontsize=7.5)

    ax = axs[1, 1]
    ax.axhline(P9999_REF, color="0.35", lw=0.9, linestyle=(0, (3, 2)), zorder=1)
    ax.plot(h, s["p9999_inst"], "o-", color=C_I, lw=1.2, ms=4)
    ax.plot(h, s["p9999_mean"], "s-", color=C_M, lw=1.2, ms=4)
    ax.axvline(NYQUIST_H, color="0.35", lw=0.9, linestyle=(0, (3, 2)), zorder=1)
    ax.set_xscale("log")
    ax.set_xticks(h)
    ax.set_xticklabels(TICK_LAB, fontsize=7)
    ax.tick_params(labelsize=7, top=False)
    styles.heading(ax=ax, letter="D",
                   heading="Sewer tracer, 99.99th percentile", fontsize=7.5)
    ax.annotate("reference", xy=(h.min(), P9999_REF), xytext=(0, 3),
                textcoords="offset points", fontsize=6.5, color="0.35")

    for ax in axs.flat:
        styles.xlabel(ax=ax, label="Coupling interval")
        ax.annotate(r"$M_2$ Nyquist", xy=(NYQUIST_H, 0.96),
                    xycoords=("data", "axes fraction"),
                    xytext=(3, 0), textcoords="offset points",
                    fontsize=6.5, color="0.35", va="top", ha="left")
    hs, ls = axs[0, 0].get_legend_handles_labels()
    styles.graph_legend(ax=axs[1, 0], handles=hs, labels=ls, loc="lower center",
                        bbox_to_anchor=(1.05, -0.42), ncol=2, frameon=False, fontsize=7.5)

### Reading the figure

Panels A--C are RMSE against the 15-minute reference. Panel D is the sewer tracer at
its 99.99th percentile, with the reference value dashed. The vertical rule is the
Nyquist limit for $M_2$, 6.21 h. Setting `GRID` at the top of the notebook redraws
all four panels from that grid's archive.

Panel A carries the result, and panel B repeats it in the sewer seepage. Below 8
hours the averaged curve sits above the instantaneous one, the damping penalty, and
both stay under 2.2 mm. At daily coupling they part by an order of magnitude in the
other direction, the aliasing penalty. Both features hold on all three grids.

The separation does not begin at the Nyquist limit, and it is not monotonic. At 8
hours, already above the limit, the two curves are still together. At 12 hours the
sampled curve stands above its own value at 24 hours, because both intervals alias
$M_2$ onto the same 14.8-day beat while 8 hours aliases it onto a 22.5-hour beat that
the aquifer damps. The limit explains the mechanism; the alias period explains the
size.

Panels C and D are flat. The tracer field does not respond to the reduction at any
interval on the coarse or midres grids, and on highres it carries a 3 to 4 percent
offset at intervals where the head has already converged, which is a noise floor
rather than a signal. Panel D is drawn from the upper tail for that reason: the
maximum is a single cell -- 1669 on the coarse grid in every run -- and it moves by
tens of percent there while the field around it, and both other grids, do not move at
all. The manuscript figure still plots that maximum in its panel D; read together,
the difference between the two versions is the difference between a hot spot and the
domain.

So the two reductions fail by different mechanisms and the tide decides which is
active, but only the aquifer response is sensitive enough to show it. The tracer at
these coupling intervals cannot distinguish them, and should not be asked to.